In [133]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

in my terminal i examined the file using the command: zcat accepted_2007_to_2018Q4.csv.gz | head -n to get columns

In [134]:
categorical = ["term","grade","sub_grade","emp_length","home_ownership","loan_status","purpose"]

numerical = ["loan_amnt","int_rate","installment","annual_inc","dti","fico_range_low",
       "fico_range_high","num_tl_90g_dpd_24m","delinq_2yrs","revol_util",
       "total_acc","open_acc","out_prncp","total_pymnt"]

In [135]:
dtypes = { **{c: 'category' for c in categorical}, **{c: 'float32' for c in numerical} }

In [136]:
df = pd.read_csv(
    "Data/accepted_2007_to_2018Q4.csv.gz",
    usecols=categorical + numerical + ['issue_d'],
    dtype=dtypes,
    parse_dates=['issue_d'],
    date_format='%b-%Y',
)

df.to_parquet('Data/accepted.parquet')

In [137]:
df_pd = pd.read_parquet("Data/accepted.parquet")

In [138]:
db = duckdb.sql

In [139]:
db("""
DESCRIBE "Data/accepted.parquet";
""").df()

,column_name,column_type,null,key,default,extra
0,loan_amnt,FLOAT,YES,None,None,None
1,term,VARCHAR,YES,None,None,None
2,int_rate,FLOAT,YES,None,None,None
3,installment,FLOAT,YES,None,None,None
4,grade,VARCHAR,YES,None,None,None
5,sub_grade,VARCHAR,YES,None,None,None
6,emp_length,VARCHAR,YES,None,None,None
7,home_ownership,VARCHAR,YES,None,None,None
8,annual_inc,FLOAT,YES,None,None,None
9,issue_d,TIMESTAMP,YES,None,None,None


In [140]:
db("""
SELECT COUNT(*) AS total_rows
FROM 'Data/accepted.parquet';
""").df()

,total_rows
0,2260701


In [141]:
db("""
SELECT *
FROM 'Data/accepted.parquet'
LIMIT 5;
""").df()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,issue_d,...,dti,delinq_2yrs,fico_range_low,fico_range_high,open_acc,revol_util,total_acc,out_prncp,total_pymnt,num_tl_90g_dpd_24m
0,3600.0,36 months,13.990000,123.029999,C,C4,10+ years,MORTGAGE,55000.0,2015-12-01,...,5.910000,0.0,675.0,679.0,7.0,29.700001,13.0,0.000000,4421.724121,0.0
1,24700.0,36 months,11.990000,820.280029,C,C1,10+ years,MORTGAGE,65000.0,2015-12-01,...,16.059999,1.0,715.0,719.0,22.0,19.200001,38.0,0.000000,25679.660156,0.0
2,20000.0,60 months,10.780000,432.660004,B,B4,10+ years,MORTGAGE,63000.0,2015-12-01,...,10.780000,0.0,695.0,699.0,6.0,56.200001,18.0,0.000000,22705.923828,0.0
3,35000.0,60 months,14.850000,829.900024,C,C5,10+ years,MORTGAGE,110000.0,2015-12-01,...,17.059999,0.0,785.0,789.0,13.0,11.600000,17.0,15897.650391,31464.009766,0.0
4,10400.0,60 months,22.450001,289.910004,F,F1,3 years,MORTGAGE,104433.0,2015-12-01,...,25.370001,1.0,695.0,699.0,12.0,64.500000,35.0,0.000000,11740.500000,0.0


In [142]:
null = db("""
SELECT COUNT(*)  AS Total_Count, COUNT(*) - COUNT(COLUMNS(*))
FROM "Data/accepted.parquet";
""").df().iloc[0]
null

Total_Count           2260701
loan_amnt                  33
term                       33
int_rate                   33
installment                33
grade                      33
sub_grade                  33
emp_length             146940
home_ownership             33
annual_inc                 37
issue_d                    33
loan_status                33
purpose                    33
dti                      1744
delinq_2yrs                62
fico_range_low             33
fico_range_high            33
open_acc                   62
revol_util               1835
total_acc                  62
out_prncp                  33
total_pymnt                33
num_tl_90g_dpd_24m      70309
Name: 0, dtype: int64

we can see multiple cols having the same number of nulls, we shall inspect that

In [143]:
similar_nulls = null[null <= 33].index.tolist()

df = db("""
SELECT * FROM "Data/accepted.parquet"
""").df()[similar_nulls]

In [144]:
similar_nulls

['loan_amnt',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'home_ownership',
 'issue_d',
 'loan_status',
 'purpose',
 'fico_range_low',
 'fico_range_high',
 'out_prncp',
 'total_pymnt']

In [145]:
#checking if the nulls of these cols are the same in all same rows
mask = df[similar_nulls].isna().all(axis=1)
print(mask.sum())                    # if we see 33, it means yes

33


In [146]:
#we'll visually look at the NaN's over the 14 columns and decide
junk = df[df['loan_amnt'].isna()]
print(junk.head(33))
print(junk.notna().sum())            #to see which columns still hold values in these rows

         loan_amnt term  int_rate  installment grade sub_grade home_ownership  \
421095         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
421096         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
528961         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
528962         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
651664         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
651665         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
749520         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
749521         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
877716         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
877717         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
983169         NaN  NaN       NaN          NaN   NaN       NaN            NaN   
983170         NaN  NaN     

since our intuition is confirmed about theses nulls in theses 14 cols, we'll drop them as they don't serve any purpose, if there were some of the cols that are nans only there could be a pattern but all of the 14 are empty, nothing to extract from it, we'll only depend on the others that holds values and can give insight.

In [166]:
#FILT_TABLE = FILTERED TABLE
db('TRUNCATE FILT_TABLE')
db (""" DROP TABLE FILT_TABLE""") #sometimes i execute the whole code at once, this is just to avoid the error concerning the exitence of the table
db("""
CREATE TABLE FILT_TABLE AS
SELECT * FROM "Data/accepted.parquet"
WHERE NOT (loan_amnt IS NULL AND term IS NULL AND int_rate IS NULL
  AND installment IS NULL AND grade IS NULL AND sub_grade IS NULL
  AND home_ownership IS NULL AND issue_d IS NULL AND loan_status IS NULL
  AND purpose IS NULL AND fico_range_low IS NULL AND fico_range_high IS NULL
  AND out_prncp IS NULL AND total_pymnt IS NULL);
""").df()

,Count
0,2260668


In [167]:
db('DESCRIBE FILT_TABLE')

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
loan_amnt,FLOAT,YES,NULL,NULL,NULL
term,VARCHAR,YES,NULL,NULL,NULL
int_rate,FLOAT,YES,NULL,NULL,NULL
installment,FLOAT,YES,NULL,NULL,NULL
grade,VARCHAR,YES,NULL,NULL,NULL
sub_grade,VARCHAR,YES,NULL,NULL,NULL
emp_length,VARCHAR,YES,NULL,NULL,NULL
home_ownership,VARCHAR,YES,NULL,NULL,NULL
annual_inc,FLOAT,YES,NULL,NULL,NULL
issue_d,TIMESTAMP,YES,NULL,NULL,NULL


In [168]:
cols = ['emp_length', 'issue_d', "dti", 'delinq_2yrs', 'num_tl_90g_dpd_24m']
new_cols = ['employment_length', 'Date', 'debt_t_ratio', 'delinquency_past_2', 'd90_delinq_past_24m'] 
for old, new in zip(cols, new_cols):
    db(f"ALTER TABLE FILT_TABLE RENAME COLUMN {old} TO {new}")

In [169]:
db("""SELECT * FROM FILT_TABLE
LIMIT 5""")

loan_amntFLOAT,termVARCHAR,int_rateFLOAT,installmentFLOAT,gradeVARCHAR,sub_gradeVARCHAR,employment_lengthVARCHAR,home_ownershipVARCHAR,annual_incFLOAT,DateTIMESTAMP,loan_statusVARCHAR,purposeVARCHAR,debt_t_ratioFLOAT,delinquency_past_2FLOAT,fico_range_lowFLOAT,fico_range_highFLOAT,open_accFLOAT,revol_utilFLOAT,total_accFLOAT,out_prncpFLOAT,total_pymntFLOAT,d90_delinq_past_24mFLOAT
3600.0,36 months,13.989999771118164,123.02999877929688,C,C4,10+ years,MORTGAGE,55000.0,2015-12-01 00:00:00,Fully Paid,debt_consolidation,5.909999847412109,0.0,675.0,679.0,7.0,29.700000762939453,13.0,0.0,4421.72412109375,0.0
24700.0,36 months,11.989999771118164,820.280029296875,C,C1,10+ years,MORTGAGE,65000.0,2015-12-01 00:00:00,Fully Paid,small_business,16.059999465942383,1.0,715.0,719.0,22.0,19.200000762939453,38.0,0.0,25679.66015625,0.0
20000.0,60 months,10.779999732971191,432.6600036621094,B,B4,10+ years,MORTGAGE,63000.0,2015-12-01 00:00:00,Fully Paid,home_improvement,10.779999732971191,0.0,695.0,699.0,6.0,56.20000076293945,18.0,0.0,22705.923828125,0.0
35000.0,60 months,14.850000381469727,829.9000244140625,C,C5,10+ years,MORTGAGE,110000.0,2015-12-01 00:00:00,Current,debt_consolidation,17.059999465942383,0.0,785.0,789.0,13.0,11.600000381469727,17.0,15897.650390625,31464.009765625,0.0
10400.0,60 months,22.450000762939453,289.9100036621094,F,F1,3 years,MORTGAGE,104433.0,2015-12-01 00:00:00,Fully Paid,major_purchase,25.3700008392334,1.0,695.0,699.0,12.0,64.5,35.0,0.0,11740.5,0.0


In [170]:
fil_table_df = db("""SELECT * FROM FILT_TABLE""").df()

In [171]:
fil_table_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 22 columns):
 #   Column               Dtype         
---  ------               -----         
 0   loan_amnt            float32       
 1   term                 str           
 2   int_rate             float32       
 3   installment          float32       
 4   grade                str           
 5   sub_grade            str           
 6   employment_length    str           
 7   home_ownership       str           
 8   annual_inc           float32       
 9   Date                 datetime64[us]
 10  loan_status          str           
 11  purpose              str           
 12  debt_t_ratio         float32       
 13  delinquency_past_2   float32       
 14  fico_range_low       float32       
 15  fico_range_high      float32       
 16  open_acc             float32       
 17  revol_util           float32       
 18  total_acc            float32       
 19  out_prncp            float32    

In [172]:
fil_table_df.duplicated().sum()

np.int64(0)

In [173]:
fil_table_df.isna().sum()

loan_amnt                   0
term                        0
int_rate                    0
installment                 0
grade                       0
sub_grade                   0
employment_length      146907
home_ownership              0
annual_inc                  4
Date                        0
loan_status                 0
purpose                     0
debt_t_ratio             1711
delinquency_past_2         29
fico_range_low              0
fico_range_high             0
open_acc                   29
revol_util               1802
total_acc                  29
out_prncp                   0
total_pymnt                 0
d90_delinq_past_24m     70276
dtype: int64

In [175]:
db("""
SELECT d90_delinq_past_24m, Date
FROM FILT_TABLE
""")

d90_delinq_past_24mFLOAT,DateTIMESTAMP
0.0,2015-12-01 00:00:00
0.0,2015-12-01 00:00:00
0.0,2015-12-01 00:00:00
0.0,2015-12-01 00:00:00
0.0,2015-12-01 00:00:00
0.0,2015-12-01 00:00:00
0.0,2015-12-01 00:00:00
1.0,2015-12-01 00:00:00
0.0,2015-12-01 00:00:00
0.0,2015-12-01 00:00:00


In [177]:
fil_table_df[['Date', 'd90_delinq_past_24m']]

,Date,d90_delinq_past_24m
0,2015-12-01,0.0
1,2015-12-01,0.0
2,2015-12-01,0.0
3,2015-12-01,0.0
4,2015-12-01,0.0
...,...,...
2260663,2016-10-01,0.0
2260664,2016-10-01,0.0
2260665,2016-10-01,7.0
2260666,2016-10-01,0.0
